## Feature generation

In [2]:
import pandas as pd
import numpy as np
import skfp.fingerprints as sfp
from skfp.fingerprints import PhysiochemicalPropertiesFingerprint

C:\Users\adamk\Desktop\politechnika\sem5\problem_classes\visco\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [82]:
visco = pd.read_csv('data/viscoelastic_cleared.csv', keep_default_na=False, na_values='')

In [4]:
visco.info()

<class 'pandas.DataFrame'>
RangeIndex: 8152 entries, 0 to 8151
Data columns (total 21 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id_surfactant1      8152 non-null   str    
 1   type_surfactant1    8152 non-null   str    
 2   smiles_surfactant1  8152 non-null   str    
 3   c_surfactant1       8152 non-null   float64
 4   id_surfactant2      8152 non-null   str    
 5   type_surfactant2    8152 non-null   str    
 6   smiles_surfactant2  8152 non-null   str    
 7   c_surfactant2       2726 non-null   float64
 8   id_additive1        8152 non-null   str    
 9   smiles_additive1    8152 non-null   str    
 10  c_additive1         5522 non-null   float64
 11  id_additive2        8152 non-null   str    
 12  smiles_additive2    8152 non-null   str    
 13  c_additive2         1333 non-null   float64
 14  temperature         8152 non-null   int64  
 15  viscoelasticity     8152 non-null   float64
 16  g_0              

In [5]:
visco.head()

,id_surfactant1,type_surfactant1,smiles_surfactant1,c_surfactant1,id_surfactant2,type_surfactant2,smiles_surfactant2,c_surfactant2,id_additive1,smiles_additive1,...,id_additive2,smiles_additive2,c_additive2,temperature,viscoelasticity,g_0,g_min,omega_cr,tau_r,eta_0
0,SLES,anionic,CCCCCCCCCCCCOCCOS(=O)(=O)[O-].[Na+],395.0,LED1A,zwitterionic,CCCCCCCCCCCC(=O)NCCC[N+](C)C(=O)[O-],55.0,NaCl,[Cl-].[Na+],...,None,None,NaN,25,0.0,NaN,NaN,NaN,NaN,0.02
1,SLES,anionic,CCCCCCCCCCCCOCCOS(=O)(=O)[O-].[Na+],395.0,LED1A,zwitterionic,CCCCCCCCCCCC(=O)NCCC[N+](C)C(=O)[O-],55.0,NaCl,[Cl-].[Na+],...,None,None,NaN,25,1.0,4.0,2.70,150.0,0.01,0.03
2,SLES,anionic,CCCCCCCCCCCCOCCOS(=O)(=O)[O-].[Na+],395.0,LED1A,zwitterionic,CCCCCCCCCCCC(=O)NCCC[N+](C)C(=O)[O-],55.0,NaCl,[Cl-].[Na+],...,None,None,NaN,25,1.0,4.5,1.20,10.0,0.10,0.45
3,SLES,anionic,CCCCCCCCCCCCOCCOS(=O)(=O)[O-].[Na+],395.0,LED1A,zwitterionic,CCCCCCCCCCCC(=O)NCCC[N+](C)C(=O)[O-],55.0,NaCl,[Cl-].[Na+],...,None,None,NaN,25,1.0,5.0,0.50,0.8,1.25,6.25
4,SLES,anionic,CCCCCCCCCCCCOCCOS(=O)(=O)[O-].[Na+],395.0,LED1A,zwitterionic,CCCCCCCCCCCC(=O)NCCC[N+](C)C(=O)[O-],55.0,NaCl,[Cl-].[Na+],...,None,None,NaN,25,1.0,4.2,0.35,1.5,0.67,2.80


In [60]:
smiles_surfactant = visco['smiles_surfactant1']
smiles_surfactant = pd.concat([smiles_surfactant, visco['smiles_surfactant2']]).drop_duplicates()
smiles_surfactant = smiles_surfactant.replace('None', np.nan).dropna()
len(smiles_surfactant)

80

In [70]:
fp = PhysiochemicalPropertiesFingerprint(fp_size=256)
surfactant_fingerprints = fp.transform(smiles_surfactant.values)

In [92]:
fingerprint_lookup = {
    smiles: fp for smiles, fp in zip(smiles_surfactant, surfactant_fingerprints)
}

# fingerprint_lookup['None'] = np.zeros(256)
#
# surfactant1_fp = np.stack(
#     visco["smiles_surfactant1"].map(fingerprint_lookup).to_numpy()
# )
#
# surfactant2_fp = np.stack(
#     visco["smiles_surfactant2"].map(fingerprint_lookup).to_numpy()
# )
#
# fp_df1 = pd.DataFrame(
#     surfactant1_fp,
#     index=visco.index,
#     columns=[f"fp_{i}" for i in range(256)]
# )
#
# fp_df2 = pd.DataFrame(
#     surfactant2_fp,
#     index=visco.index,
#     columns=[f"fp_{i}" for i in range(256, 512)]
# )
#
# visco = pd.concat([visco, fp_df1, fp_df2], axis=1)

In [94]:
physio = pd.DataFrame(smiles_surfactant, columns=['smiles'])

surfactant_fp = np.stack(
    physio["smiles"].map(fingerprint_lookup).to_numpy()
)

fp_df = pd.DataFrame(
    surfactant_fp,
    index=physio.index,
    columns=[f"fp_{i}" for i in range(256)]
)

physio = pd.concat([physio, fp_df], axis=1)

In [95]:
physio.info()

<class 'pandas.DataFrame'>
Index: 80 entries, 0 to 8102
Columns: 257 entries, smiles to fp_255
dtypes: str(1), uint8(256)
memory usage: 21.2 KB


In [97]:
physio.to_csv('data/viscoelastic_physiochemical.csv', index=False)